# Exploration of various ways to identify the reaches from the distance-to-kinect data


In [ ]:
### imports and globals 

import pyxdf

# for the tests
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

from scipy.signal import butter, sosfiltfilt, find_peaks

%matplotlib qt

# optional visualizations within functions
do_visualize = True

# Utility functions

## Not specific to the rearm data analysis, but useful for most exploratory the analyses

In [ ]:
# print the items of a dictionary
# useful when I do not remember the keys...
def print_items(d):
    """Print the items in a dictionary"""
    for k, v in d.items():
        print(f"{k}: {v}")

## Functions to manage rearm xdf files

In [ ]:
import sys

# add the path to the src directory to the sys.path...
sys.path.append("..")
# ... so that we can explicitly import all functions (useful for ruff)
from src.rearm.xdf import (
    print_streams_types_and_names,
    get_stream,
    get_kinect_channel,
    get_kinect_channel_index,
    get_kinect_channel_data,
)


def interpolate_to_constant_time_step(t, x, dt=0.033):
    """Interpolate the data to a constant time step"""

    n_columns = x.shape[1] if x.ndim > 1 else 1

    t_new = np.arange(t[0], t[-1], dt)

    if x.ndim < 2:
        x_new = np.interp(t_new, t, x)
    else:
        x_new = np.zeros((len(t_new), n_columns))
        for i in range(n_columns):
            x_new[:, i] = np.interp(t_new, t, x[:, i])

    return x_new, t_new

## Load the xdf file

In [ ]:
xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20211116_V3/ReArm_C1P02_20210715_V3_Reaching/ReArm_C1P07_20211116_V3_r.xdf"  # unexpected sequence - no solution
# xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V1/Reaching/task-V1_Reach.xdf"  # no mouse data --> eventIDE
# xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V2/Reaching/task-V2_Reach.xdf"  # missing second half of the session - no solution
xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P04/V1/Reaching/C01P04_QueAla_20210531_1_r.xdf"  # no mouse marker csv file associated -> nan time correction - solution found
# xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P01/V1/Reaching/001_BenMus_20210202_1_r.xdf"  # large block of zeros in the beginning -> incomplete data - no solution
# xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P45/V2/Reaching/task-V2_Reach.xdf"  # no mouse data --> eventIDE -- 1 extra reach
# xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P31/V2/Reaching/C1P31_RauMic_20230331_2_r.xdf"  # wrong correction? look good
# xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P31/V3/Reaching/task-V3_Reach.xdf"  # 3 unknown reaches - solved
# xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Reaching/ReArm_C1P42_20240603_V1_r.xdf"  # Kinect markers with 1 value that is empty + data meaningless - no solution
# xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P42/V1/Reaching/task-V1_Reach_old1.xdf" # action stopped early: missing second part of the session - no solution
# xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P42/V1/Reaching/task-V1_Reach_old2.xdf"  # only 7 second of data - no solution
# xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P42/V1/Reaching/task-V1_Reach_old3.xdf" # only about 7 second of data - no solution
# xdf_fullFname = "../dat/ReArm.lnk/PANU/ReArm_C1P05/ReArm_C1P05_20210514_V1/ReArm_C1P05_20210514_V1_Reaching/ReArm_C1P05_20210621_V1_r.xdf"  # Test file for panu identification

# xdf_fullFname = "../dat/ReArm.lnk/PANU/ReArm_C1P38/ReArm_C1P38_20231009_V1/ReArm_C1P38_20231009_V1_Reaching/ReArm_C1P38_20231009_V1_r.xdf"  # Test file for panu identification
# xdf_fullFname = "../dat/ReArm.lnk/PANU/ReArm_C1P38/ReArm_C1P38_20231027_V2/ReArm_C1P38_20231027_V2_Reaching/ReArm_C1P38_20231027_V2_r.xdf"  # kinect data bugged - no solution
# xdf_fullFname = "../dat/ReArm.lnk/PANU/ReArm_C1P38/ReArm_C1P38_20240207_V3/ReArm_C1P38_20240207_V3_Reaching/ReArm_C1P38_20240207_V3_r.xdf"  # kinect data bugged - no solution
# xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P08/V2/Reaching/C1P08_FabMar_20210903_2_r.xdf"

# xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P07/V3/Reaching/C1P07_MagPau_20211116_3_r.xdf" -- sequence problematic
xdf_data, header = pyxdf.load_xdf(
    filename=xdf_fullFname,
    select_streams=[
        {"type": "MoCap"},
        {"type": "Markers"},
    ],
    synchronize_clocks=True,
    dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
    verbose=False,
)

print_streams_types_and_names(xdf_data)

## Make the kinect time correction (if needed)

Any `*.xdf` file needing a correction of the kinect timestamps is accompanied by a `*_xdf_time_correction.csv` file with the time correction. 
If no such a file is present, no correction is needed.

In [ ]:
def get_time_correction(xdf_fullFname):
    """Get the time correction from the xdf file name"""

    is_log_present = False
    log_fullFname = os.path.join(
        os.path.dirname(xdf_fullFname), "..", "kinect_time_correction.log"
    )
    log_fullFname = os.path.normpath(log_fullFname)
    if os.path.exists(log_fullFname):
        is_log_present = True

    is_correction_present = False
    time_correction_file = xdf_fullFname.replace(".xdf", "_xdf_time_correction.csv")
    if os.path.exists(time_correction_file):
        is_correction_present = True

    if not is_log_present:
        raise FileNotFoundError(
            f"The file {log_fullFname} does not exist. Please run the script to generate it."
        )

    if not is_correction_present:
        time_correction = np.float64(0)
    else:
        time_correction = np.loadtxt(time_correction_file, delimiter=",", skiprows=1)

    return time_correction


time_correction = get_time_correction(xdf_fullFname)
print(f"Time correction: {time_correction} s")

if not np.isnan(time_correction):
    # make the time correction
    kinect_mocap = get_stream(xdf_data, "MoCap", "EuroMov-Mocap-Kinect")
    kinect_markers = get_stream(xdf_data, "Markers", "EuroMov-Markers-Kinect")

    if kinect_mocap:
        kinect_mocap["time_stamps"] = kinect_mocap["time_stamps"] + time_correction
    if kinect_markers:
        kinect_markers["time_stamps"] = kinect_markers["time_stamps"] + time_correction

# Remove the kinect samples filled with zeros
This has to be done before any processing of the kinect data, as this is to fix a bug due to the kinect. 

In [ ]:
kinect_mocap = get_stream(xdf_data, "MoCap", "EuroMov-Mocap-Kinect")
mouse_mocap = get_stream(xdf_data, "MoCap", "Mouse")

if kinect_mocap:
    kinect_t = kinect_mocap["time_stamps"]
    kinect_data = kinect_mocap["time_series"]
    # find the indexes of kinect data that are filled with zeros
    zero_rows = np.all(kinect_data == 0, axis=1)
    zero_rows_indices = np.where(zero_rows)[0]
    print(f"Found {len(zero_rows_indices)} rows filled with only zeros")
    if len(zero_rows_indices) > 0:
        # remove the zero rows from the data
        kinect_data = np.delete(kinect_data, zero_rows_indices, axis=0)
        kinect_t = np.delete(kinect_t, zero_rows_indices, axis=0)

        WristRight_Z_before = get_kinect_channel_data(kinect_mocap, "WristRight_Z")
        kinect_t_before = kinect_mocap["time_stamps"]

        # modify the original data
        kinect_mocap["time_series"] = kinect_data
        kinect_mocap["time_stamps"] = kinect_t

        WristRight_Z = get_kinect_channel_data(kinect_mocap, "WristRight_Z")

        # plot WristRight_Z
        plt.figure()
        plt.plot(kinect_t_before, WristRight_Z_before, ".", label="before")
        plt.plot(kinect_t, WristRight_Z, ".", label="after")
        plt.title("WristRight_Z: before and after removing zero rows")
        plt.xlabel("Time (s)")
        plt.ylabel("Position (m)")
        plt.legend()
        plt.show()

        del WristRight_Z_before
        del kinect_t_before

    # clean globals
    del zero_rows
    del zero_rows_indices


# Interpolate the Mocap data
This is mandatory because the kinect and mouse data are produced by the computer: the sampling rate is not waranted to be constant (samples are forgetten... sometimes). 

In [ ]:
def get_joint_norm(kinect_mocap, joint_name):
    """Get the norm of the joint position"""
    joint_X = get_kinect_channel_data(kinect_mocap, joint_name + "_X")
    joint_Y = get_kinect_channel_data(kinect_mocap, joint_name + "_Y")
    joint_Z = get_kinect_channel_data(kinect_mocap, joint_name + "_Z")
    joint_Norm = np.sqrt(joint_X**2 + joint_Y**2 + joint_Z**2)
    return joint_Norm


## proceed step by step with visualization
if kinect_mocap:

    # re-read the data (that may have been modified)
    kinect_t = kinect_mocap["time_stamps"]
    kinect_data = kinect_mocap["time_series"]

    # # interpolate all the kinect data to a constant time step

    WristLeft_Norm = get_joint_norm(kinect_mocap, "WristLeft")
    WristRight_Norm = get_joint_norm(kinect_mocap, "WristRight")

    reach_left, reach_t = interpolate_to_constant_time_step(kinect_t, WristLeft_Norm)
    reach_right, reach_t = interpolate_to_constant_time_step(kinect_t, WristRight_Norm)

    # Test : interpolate 2 columns at a time
    w_lr = np.vstack((WristLeft_Norm, WristRight_Norm)).T
    w_lr_, reach_t_ = interpolate_to_constant_time_step(kinect_t, w_lr)
    reach_left_ = w_lr_[:, 0]
    reach_right_ = w_lr_[:, 1]
    # assert that the result is the same
    assert reach_left.shape == reach_left_.shape
    assert reach_right.shape == reach_right_.shape
    assert reach_t.shape == reach_t_.shape
    assert len(reach_left) == len(reach_right)
    assert len(reach_left) == len(reach_t)
    assert len(reach_right) == len(reach_t)
    assert np.allclose(reach_t, reach_t_)
    assert np.allclose(reach_left, reach_left_)
    assert np.allclose(reach_right, reach_right_)
    print("Interpolation multiple column test passed")

    # plot the raw and interpolated data
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(kinect_t, WristLeft_Norm, ".", label="Left wrist", color="b")
    ax.plot(kinect_t, WristRight_Norm, ".", label="Right wrist", color="k")
    ax.plot(reach_t, reach_left, label="Left wrist interpolated", color="b", alpha=0.2)
    ax.plot(
        reach_t, reach_right, label="Right wrist interpolated", color="k", alpha=0.2
    )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Distance from the kinect (m)")
    ax.legend()
    plt.show()

    # delete the variables that MUST be recomputed
    del kinect_t
    del kinect_data
    del WristLeft_Norm
    del WristRight_Norm
    del reach_left
    del reach_right
    del reach_t
    del w_lr
    del w_lr_
    del reach_t_


#######################
## real job is below...
def resample_stream(stream):
    """Resample the stream to a constant time step"""

    # get the type and name of the stream
    stream_type = stream["info"]["type"][0]
    stream_name = stream["info"]["name"][0]
    if stream_type != "MoCap":
        return stream

    # check if the stream was resampled before
    if "resampled" in stream["info"]:
        print(f"Stream '{stream_name}' was already resampled")
        return stream

    t = stream["time_stamps"]
    data = stream["time_series"]
    # resample the data to a constant time step
    t_step = 1 / 30  # 30 Hz typical for kinect
    data, t = interpolate_to_constant_time_step(t, data, dt=t_step)
    # modify the original data
    stream["time_series"] = data
    stream["time_stamps"] = t
    stream["info"]["desc"][0]["channels"][0]["channel"][0]["nominal_srate"] = 30
    stream["info"]["desc"][0]["channels"][0]["channel"][0]["preferred_srate"] = 30
    # add a new field to the info
    stream["info"]["resampled"] = ["True"]

    return stream


if mouse_mocap:
    mouse_mocap = resample_stream(mouse_mocap)


if kinect_mocap:
    kinect_mocap = resample_stream(kinect_mocap)

    # re-read the data (that may have been modified)
    kinect_t = kinect_mocap["time_stamps"]
    kinect_data = kinect_mocap["time_series"]

# Low pass filter the data to guess the reaches approximate position
We choose a cutoff frequency of 0.5 Hz, which corresponds to a time constant of 2 seconds = typical time of a reach.

In [ ]:
def butter_lowpass(cutoff, fs, order=2):
    """Design a lowpass Butterworth filter."""
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    sos = butter(
        order,
        normal_cutoff,
        btype="low",
        output="sos",  # recommended for numerical stability by scipy
    )
    return sos


def lowpass_filter(t, data, cutoff=0.5):
    """Apply a lowpass filter to the data"""
    # check that the sampling period is constant
    dt = np.mean(np.diff(t))
    if not np.allclose(np.diff(t), dt):
        raise ValueError("The time vector do not have a constant sampling period")

    fs = 1 / dt  # sample rate, Hz
    order = 4  # order of the filter
    sos = butter_lowpass(cutoff, fs, order=order)
    filtered_data = sosfiltfilt(sos, data)
    return filtered_data

# Get the action zones from the markers
The participant is asked to follow a timed sequence: 
- Action: perform a series of reaches (20s)
- Pause: wait for the next action (20s)
- ...
- Action: perform a series of reaches (20s)
- Pause: wait for the next action (20s)


NB: Outside action zones, the data is not relevant for the reaches analysis, *and it adds a lot of noise to the data*.

The markers are necessary to identify the action zones, within which the reaches (are expected to) occur. 

Each action zone has a *start* and *end* marker, but the label of the markers differs if the sequence was generated by : 

- the software **LSL-Mouse**: stream `mouse_to_nic_markers`
    - start = `"[111]"`  
    - stop = `"[100]"` 

- the software **event-IDE**: stream `event_to_nic_markers`
    - start = `"[100]"`, but we have to keep only the first start in the sequence before each stop
    - stop = `"[75]"`


The streams are loaded from the xdf file using the function `get_stream`:
``` python
# get the markers streams from the xdf file
mouse_to_nic_markers = get_stream(xdf_data, "Markers", ["MouseToNIC"])
event_to_nic_markers = get_stream(xdf_data, "Markers", ["event_ide_TONIC"]) 
```
``` python

## Function to get the markers of the action zones in the data

In [ ]:
def find_marker_indexes(marker_name, markers_data):
    """Find the indexes of marker_name in markers_data"""

    marker_index_list = [
        i for i, marker in enumerate(markers_data) if marker_name in marker[0]
    ]

    return np.array(marker_index_list)


def get_coherent_actions_start_stop_times(starts, stops):
    """Get the coherent start and stop times of the actions"""

    # make an array of start, 0 and stop, 1
    start = np.zeros(
        len(starts),
        dtype=[("time", float), ("type", int)],
    )
    start["time"] = starts
    start["type"] = 0
    stop = np.zeros(
        len(stops),
        dtype=[("time", float), ("type", int)],
    )
    stop["time"] = stops
    stop["type"] = 1
    start_and_stop = np.concatenate((start, stop))
    start_and_stop = np.sort(start_and_stop, order="time")

    for i in range(len(start_and_stop) - 1):
        # keep only the last start in case of multiple contiguous start
        if start_and_stop[i]["type"] == 0 and start_and_stop[i + 1]["type"] == 0:
            start_and_stop[i + 1]["type"] = -1
        # keep only the first stop in case of multiple contiguous stop
        if start_and_stop[i]["type"] == 1 and start_and_stop[i + 1]["type"] == 1:
            start_and_stop[i + 1]["type"] = -1

    # ensure that the first is a start and the last is a stop
    if start_and_stop[0]["type"] == 1:
        start_and_stop[0]["type"] = -1
    if start_and_stop[-1]["type"] == 0:
        start_and_stop[-1]["type"] = -1

    # clean the start_and_stop array
    start_and_stop_ok = start_and_stop[start_and_stop["type"] != -1]

    starts_corrected = start_and_stop_ok["time"][start_and_stop_ok["type"] == 0]
    stops_corrected = start_and_stop_ok["time"][start_and_stop_ok["type"] == 1]

    if len(starts_corrected) != len(stops_corrected):
        raise ValueError(
            f"Number of starts ({len(starts_corrected)}) and stops ({len(stops_corrected)}) are not equal"
        )

    return starts_corrected, stops_corrected


def get_actions_start_stop_times_from_mouse_to_nic_markers(mouse_to_nic_markers):
    """get the start and stop times from mouse_to_nic_markers"""

    mouse_to_nic_markers_data = mouse_to_nic_markers["time_series"]
    mouse_to_nic_markers_time = mouse_to_nic_markers["time_stamps"]

    if not isinstance(mouse_to_nic_markers_data[0], list):
        mouse_to_nic_markers_data = [[str(x)] for x in mouse_to_nic_markers_data]

    start_marker_index_list = find_marker_indexes("[111]", mouse_to_nic_markers_data)
    stop_marker_index_list = find_marker_indexes("[100]", mouse_to_nic_markers_data)
    start_times_from_mouse_to_nic_markers = mouse_to_nic_markers_time[
        start_marker_index_list
    ]
    stop_times_from_mouse_to_nic_markers = mouse_to_nic_markers_time[
        stop_marker_index_list
    ]

    start_times_from_mouse_to_nic_markers, stop_times_from_mouse_to_nic_markers = (
        get_coherent_actions_start_stop_times(
            start_times_from_mouse_to_nic_markers,
            stop_times_from_mouse_to_nic_markers,
        )
    )

    return (
        start_times_from_mouse_to_nic_markers,
        stop_times_from_mouse_to_nic_markers,
    )


def get_actions_start_stop_times_from_event_ide_TONIC(event_ide_tonic):
    """get the start and stop times from event_ide_tonic"""

    event_ide_markers_data = event_ide_tonic["time_series"]
    event_ide_markers_time = event_ide_tonic["time_stamps"]

    if not isinstance(event_ide_markers_data[0], list):
        event_ide_markers_data = [[str(x)] for x in event_ide_markers_data]

    start_marker_index_list = find_marker_indexes("[100]", event_ide_markers_data)
    stop_marker_index_list = find_marker_indexes("[75]", event_ide_markers_data)

    start_times_from_event_ide_tonic = event_ide_markers_time[start_marker_index_list]
    stop_times_from_event_ide_tonic = event_ide_markers_time[stop_marker_index_list]

    # keep only the first start time for each stop time
    previous_stop_time = 0
    good_start_times = []
    for stop_time in stop_times_from_event_ide_tonic:
        possible_start_times = start_times_from_event_ide_tonic[
            start_times_from_event_ide_tonic < stop_time
        ]
        possible_start_times = possible_start_times[
            possible_start_times > previous_stop_time
        ]
        start_time = possible_start_times[0] if len(possible_start_times) > 0 else None
        good_start_times.append(start_time)
        previous_stop_time = stop_time
    good_start_times = np.array(good_start_times)
    good_start_times = good_start_times[
        good_start_times != None  # noqa: E711
    ]  # should be useless...

    good_start_times, stop_times_from_event_ide_tonic = (
        get_coherent_actions_start_stop_times(
            good_start_times,
            stop_times_from_event_ide_tonic,
        )
    )

    return (
        good_start_times,
        stop_times_from_event_ide_tonic,
    )


def get_actions_start_stop_times_from_mouse_markers(mouse_markers):
    """get the start and stop times of the blocks from mouse_markers"""

    # NOTE: alternative way to get the start and stop times
    # here used to check the consistency with the mouse_to_nic_markers

    mouse_markers_data = mouse_markers["time_series"]
    mouse_markers_time = mouse_markers["time_stamps"]

    start_marker_index_list = find_marker_indexes(
        "DoCycleChange:DoRecord", mouse_markers_data
    )
    stop_marker_index_list = find_marker_indexes(
        "DoCycleChange:DoPause", mouse_markers_data
    )
    start_times_from_mouse_markers = mouse_markers_time[start_marker_index_list]
    stop_times_from_mouse_markers = mouse_markers_time[stop_marker_index_list]

    start_times_from_mouse_markers, stop_times_from_mouse_markers = (
        get_coherent_actions_start_stop_times(
            start_times_from_mouse_markers,
            stop_times_from_mouse_markers,
        )
    )

    return (
        start_times_from_mouse_markers,
        stop_times_from_mouse_markers,
    )


def print_actions_start_stop_times(start_stop_times):
    """Print the start and stop times of the actions"""

    start_times, stop_times = start_stop_times
    for i in range(len(start_times)):
        print(
            f"action{i:02d}: {start_times[i]:8.2f} -> {stop_times[i]:8.2f}, Duration: {stop_times[i] - start_times[i]:5.2f}s"
        )

## Test the functions 

In [ ]:
## Test the functions

event_to_nic_markers = get_stream(xdf_data, "Markers", ["event_ide_TONIC"])
mouse_to_nic_markers = get_stream(xdf_data, "Markers", ["MouseToNIC"])

if mouse_to_nic_markers:
    mouse_to_nic_markers_data = mouse_to_nic_markers["time_series"]
    mouse_to_nic_markers_time = mouse_to_nic_markers["time_stamps"]
    start_t, stop_t = get_actions_start_stop_times_from_mouse_to_nic_markers(
        mouse_to_nic_markers
    )
    print("Mouse to NIC markers:")
    print_actions_start_stop_times((start_t, stop_t))
    # for the assert
    start_t_nic = start_t
    stop_t_nic = stop_t

if event_to_nic_markers:
    event_to_nic_markers_data = event_to_nic_markers["time_series"]
    event_to_nic_markers_time = event_to_nic_markers["time_stamps"]
    start_t, stop_t = get_actions_start_stop_times_from_event_ide_TONIC(
        event_to_nic_markers
    )
    print("Event IDE to NIC markers:")
    print_actions_start_stop_times((start_t, stop_t))

# NOTE: To verify that we get the same start and stop times from the mouse markers and the mouse to nic markers
mouse_markers = get_stream(xdf_data, "Markers", ["Mouse", "Mouse-Markers"])
if mouse_markers:
    mouse_markers_data = mouse_markers["time_series"]
    mouse_markers_time = mouse_markers["time_stamps"]
    start_t, stop_t = get_actions_start_stop_times_from_mouse_markers(mouse_markers)

    # assert that the start and stop times are the same
    start_t_mouse = start_t
    stop_t_mouse = stop_t
    assert len(start_t_mouse) == len(start_t_nic)
    assert len(stop_t_mouse) == len(stop_t_nic)
    assert np.allclose(start_t_mouse, start_t_nic)
    assert np.allclose(stop_t_mouse, stop_t_nic)
    print("**** Mouse markers and NIC markers are coherent :-) ****")


if kinect_mocap:
    kinect_t = kinect_mocap["time_stamps"]
    WristRight_X = get_kinect_channel_data(kinect_mocap, "WristRight_X")
    WristRight_Y = get_kinect_channel_data(kinect_mocap, "WristRight_Y")
    WristRight_Z = get_kinect_channel_data(kinect_mocap, "WristRight_Z")

    WristLeft_X = get_kinect_channel_data(kinect_mocap, "WristLeft_X")
    WristLeft_Y = get_kinect_channel_data(kinect_mocap, "WristLeft_Y")
    WristLeft_Z = get_kinect_channel_data(kinect_mocap, "WristLeft_Z")

    WristLeft_Norm = np.sqrt(WristLeft_X**2 + WristLeft_Y**2 + WristLeft_Z**2)
    WristRight_Norm = np.sqrt(WristRight_X**2 + WristRight_Y**2 + WristRight_Z**2)
    print(f"kinect_mocap['time_series'] shape: {kinect_mocap['time_series'].shape}")


if event_to_nic_markers:
    event_markers_data = event_to_nic_markers["time_series"]
    event_markers_time = event_to_nic_markers["time_stamps"]
    print(f"Event markers data : {event_markers_data.shape}")
    print(f"Event markers time : {event_markers_time.shape}")

## Functions to identify the reaches
Includes debug options to visualize the data and the identified reaches.

In [ ]:
def index_of_last_negative_velocity_before_peak(t, velocity, t_end_i=None):
    """Get the index of the last negative velocity before the velocity peak"""

    if t_end_i is None:
        t_end_i = np.argmin(velocity)
    t_end = t[t_end_i]
    positive_before_t_end = velocity[t < t_end] > 0
    index_before_t_end = np.where(positive_before_t_end)[0]
    index_before_t_end += 1  # NOTE: not sure why, but needed...
    if len(index_before_t_end) == 0:
        return -1
    else:
        return max(index_before_t_end)


def plot_one_reach_two_axes(
    t_beg, t_end, t_beg_i, t_end_i, t_beg_mask, t_end_mask, wrist_norm, title_txt=""
):

    t = kinect_mocap["time_stamps"]
    wrist_beg_position = wrist_norm[t_beg_mask]
    wrist_end_position = wrist_norm[t_end_mask]

    def plot_one_sub(ax):
        ax.plot(t, wrist_norm, ".", label="Position", color="b")
        ax.plot(
            t[t_beg_mask],
            wrist_norm[t_beg_mask],
            "o",
            label="Beg reach t_beg_mask",
            color="g",
        )
        ax.plot(
            t[t_end_mask],
            wrist_norm[t_end_mask],
            "o",
            label="End reach t_end_mask",
            color="r",
        )

        ax.plot(
            t_beg,
            np.median(wrist_beg_position),
            "*",
            label="Beg reach median",
            color="g",
            markersize=20,
            markeredgewidth=2,
            markeredgecolor="k",
        )
        ax.plot(
            t_end,
            np.median(wrist_end_position),
            "*",
            label="End reach median",
            color="r",
            markersize=20,
            markeredgewidth=2,
            markeredgecolor="k",
        )

        ax.vlines(
            t_beg,
            ymin=np.min(wrist_norm),
            ymax=np.max(wrist_norm),
            color="g",
            linestyle="--",
            label="Beg reach",
        )
        ax.vlines(
            t_end,
            ymin=np.min(wrist_norm),
            ymax=np.max(wrist_norm),
            color="r",
            linestyle="--",
            label="End reach",
        )

        ax.hlines(
            np.median(wrist_beg_position),
            xmin=t_beg,
            xmax=t_end,
            color="g",
            linestyle="--",
        )
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Distance from the kinect (m)")

    # plot the data with 2 subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5))

    plot_one_sub(ax1)
    ax1.set_title(title_txt)
    plot_one_sub(ax2)
    t_zoom = t_beg - 2, t_end + 2
    ax2.set_xlim(t_zoom)
    # ax2.set_title("Zoom on the reach of interest")
    from matplotlib.patches import ConnectionPatch

    xy1 = (float(t_end), float(np.min(wrist_norm)))
    xy2 = (float(t_end), float(np.max(wrist_norm)))
    connection_end = ConnectionPatch(
        xyA=xy1,
        xyB=xy2,
        coordsA="data",
        coordsB="data",
        axesA=ax1,
        axesB=ax2,
        color="red",
    )
    ax2.add_artist(connection_end)

    xy1 = (t_beg, np.min(wrist_norm))
    xy2 = (t_beg, np.max(wrist_norm))
    connection_beg = ConnectionPatch(
        xyA=xy1,
        xyB=xy2,
        coordsA="data",
        coordsB="data",
        axesA=ax1,
        axesB=ax2,
        color="green",
    )
    ax2.add_artist(connection_beg)

    plt.show()


def get_one_reach(t, wrist_norm, t_end_i, cutoff=0.5, do_plot=False, title_txt=""):
    """
    get one reach from the position and time data

    return:
    {
        "beg_position": position at the beginning of the reach,
        "end_position": position at the end of the reach,
        "t_beg": time at the beginning of the reach,
        "t_end": time at the end of the reach,
        "t_beg_i": index of the beginning of the reach,
        "t_end_i": index of the end of the reach
    }
    """
    wrist_norm = np.array(wrist_norm)
    t = np.array(t)

    wrist_norm_velocity = np.gradient(wrist_norm, t)
    filtered_wrist_norm_velocity = lowpass_filter(t, wrist_norm_velocity, cutoff=cutoff)

    # we know that the reach end
    t_end = t[t_end_i]
    # we look for the reach start
    t_beg_i = index_of_last_negative_velocity_before_peak(
        t, filtered_wrist_norm_velocity, t_end_i
    )
    t_beg = t[t_beg_i]

    # verify that t_beg_i is not too close to the t_end_i
    if t_end_i - t_beg_i < 15:  # half a second
        t_beg_i = index_of_last_negative_velocity_before_peak(
            t, filtered_wrist_norm_velocity, t_beg_i
        )

    # get the mask around t_beg_*_i +/- 10
    mask_width = 10
    t_beg_mask = range(t_beg_i - mask_width, t_beg_i + mask_width)
    t_end_mask = range(t_end_i - mask_width, t_end_i + mask_width)

    # we will return the median of the following positions
    wrist_beg_position = wrist_norm[t_beg_mask]
    wrist_end_position = wrist_norm[t_end_mask]

    # verify that the reach distance is larger than 0.05 m
    if np.abs(np.median(wrist_end_position) - np.median(wrist_beg_position)) < 0.05:
        return None

    if do_plot:
        plot_one_reach_two_axes(
            t_beg,
            t_end,
            t_beg_i,
            t_end_i,
            t_beg_mask,
            t_end_mask,
            wrist_norm,
            title_txt=title_txt,
        )
    return {
        "beg_position": np.median(wrist_beg_position),
        "end_position": np.median(wrist_end_position),
        "t_beg": t_beg,
        "t_end": t_end,
        "t_beg_i": t_beg_i,
        "t_end_i": t_end_i,
    }

## Get the reaches in the action zones


In [ ]:
def get_reaches(t, wrist, wrist_f, title_txt=""):
    """get the reaches on this wrist"""

    peaks_threshold = np.median(wrist_f) - 0.1
    i_inter_peaks = 60  # 2 seconds

    ########################################################################################
    # alternative way to find the reaches from the peaks (simpler logic)

    # find the negative peaks in the wrist that prominent of 0.1 and at least 2 seconds apart
    # NOTE: prominence works very well but it is difficult to understand the meaning of the value
    # neg_peaks, _ = find_peaks(-wrist_f, prominence=0.1, distance=60)  # 60 = 2s

    # find the negative peaks in the wrist that are below the threshold and at least 2 seconds apart
    # NOTE: this is the simplest method to find the peaks (same as method 1)
    neg_peaks, _ = find_peaks(-wrist_f, height=-peaks_threshold, distance=i_inter_peaks)

    # remove the peaks that are outliers (all reaches should end at the same target position)
    # NOTE: this is the simplest method (same as method 1)
    reaches_end = wrist_f[neg_peaks]
    reaches_end_median = np.median(reaches_end)
    reaches_end_iqr = np.percentile(reaches_end, 75) - np.percentile(reaches_end, 25)
    i_outliers = [
        i
        for i in range(len(reaches_end))
        if abs(reaches_end[i] - reaches_end_median) > 3 * reaches_end_iqr
    ]
    neg_peaks = np.delete(neg_peaks, i_outliers)

    # equivalent to index_of_last_negative_velocity_before_peak()... but simpler conceptually
    pos_peaks, _ = find_peaks(wrist_f)

    # put negative and positive peaks in the same list sorted by time
    negative_peaks = [
        {"index": neg_peaks[i], "from": "neg"} for i in range(len(neg_peaks))
    ]
    positive_peaks = [
        {"index": pos_peaks[i], "from": "pos"} for i in range(len(pos_peaks))
    ]
    pks = negative_peaks + positive_peaks
    pks = sorted(pks, key=lambda x: x["index"])

    # for each negative peak, keep only the previous positive peak
    reaches = []
    for i in range(1, len(pks)):
        if pks[i]["from"] == "neg":
            i_end = pks[i]["index"]
            i_beg = pks[i - 1]["index"]
            reach_distance = wrist_f[i_end] - wrist_f[i_beg]
            if -reach_distance > 0.05:  # 5 cm
                reaches.append(
                    {
                        "i_beg": i_beg,
                        "i_end": i_end,
                        "length": reach_distance,
                    }
                )

    # # remove the peaks that are large outliers in distance
    # # NOTE: this makes a difference with the previous method (assert does not pass)
    # # but it is best to remove the outliers in distance (they are outlier!)
    # reaches_length = np.array([r["length"] for r in reaches])
    # length_median = np.median(reaches_length)
    # length_iqr = np.percentile(reaches_length, 75) - np.percentile(reaches_length, 25)
    # i_outliers = [
    #     i
    #     for i in range(len(reaches_length))
    #     if abs(reaches_length[i] - length_median) > 3 * length_iqr
    # ]
    # reaches = np.delete(reaches, i_outliers)

    reaches_2 = reaches.copy()

    # plot the peaks
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(t, wrist_f, ".", label="Wrist position", color="b")
    for r in reaches_2:
        ax.plot(
            t[r["i_beg"]],
            wrist_f[r["i_beg"]],
            "o",
            label="Beg reach",
            color="g",
        )
        ax.plot(
            t[r["i_end"]],
            wrist_f[r["i_end"]],
            "o",
            label="End reach",
            color="r",
        )

    plt.show()
    # return None, None, None
    ########################################################################################

    # find the negative peaks in the wrist that are below the threshold and at least 2 seconds apart
    peaks, _ = find_peaks(-wrist_f, height=-peaks_threshold, distance=i_inter_peaks)
    if len(peaks) == 0:
        print("No peaks found")
        return None, None, None

    # remove the peaks that are outliers (all reaches should end at the same target position)
    reaches_end = wrist_f[peaks]
    reaches_end_median = np.median(reaches_end)
    reaches_end_iqr = np.percentile(reaches_end, 75) - np.percentile(reaches_end, 25)
    i_outliers = [
        i
        for i in range(len(reaches_end))
        if abs(reaches_end[i] - reaches_end_median) > 3 * reaches_end_iqr
    ]
    peaks = np.delete(peaks, i_outliers)

    # get the reaches (t_beg, t_end)
    reaches = []
    for pk in peaks:
        reach = get_one_reach(
            t,
            wrist,
            t_end_i=pk,
            do_plot=True if pk == peaks[0] else False,
            title_txt=f"Reach {len(reaches)+1} / {len(peaks)} - {title_txt}",
        )
        if reach:
            reaches.append(reach)

    ## plot the reaches and the reaches_2
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(t, wrist_f, ".", label="Wrist position", color="b")
    for r in reaches:
        ax.plot(
            t[r["t_end_i"]],
            wrist_f[r["t_end_i"]],
            "*",
            label="End reach",
            color="r",
            markersize=15,
        )
        ax.plot(
            t[r["t_beg_i"]],
            wrist_f[r["t_beg_i"]],
            "*",
            label="Beg reach",
            color="g",
            markersize=15,
        )
    # plot it after, as we might have removed some outliers
    for r in reaches_2:
        ax.plot(
            t[r["i_end"]],
            wrist_f[r["i_end"]],
            "o",
            label="End reach (peaks)",
            color="green",
        )
        ax.plot(
            t[r["i_beg"]],
            wrist_f[r["i_beg"]],
            "o",
            label="Beg reach (peaks)",
            color="orange",
        )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Distance from the kinect (m)")
    plt.title(f"{len(reaches)} : {len(reaches_2)} reaches detected")
    plt.show()

    # if len(reaches) != len(reaches_2):
    #     print(
    #         f"Method 2 removed {len(reaches) - len(reaches_2)} outliers in the reaches (peaks) detection"
    #     )
    # else:
    if True:
        assert len(reaches) == len(
            reaches_2
        ), "The two methods should return the same number of reaches"

        assert np.allclose(
            [r["t_end_i"] for r in reaches],
            [r["i_end"] for r in reaches_2],
        ), "The two methods should return the same indexes of reaches END"

        t_start_1 = [r["t_beg_i"] for r in reaches]
        t_start_2 = [r["i_beg"] for r in reaches_2]

        error_in_start = np.array(t_start_1) - np.array(t_start_2)
        if np.max(abs(error_in_start)) > 1:

            fig, ax = plt.subplots(figsize=(10, 5))
            ax.boxplot(
                error_in_start,
                positions=[0],
                widths=0.5,
            )
            ax.set_title("Boxplot of the start times of the reaches")
            ax.set_ylabel("Time (s)")
            ax.set_xlabel("Method")
            plt.show()

        assert np.allclose(
            [r["t_beg_i"] for r in reaches],
            [r["i_beg"] for r in reaches_2],
            atol=1,  # 1 sample is allowed
        ), "The two methods should return the same indexes of reaches START"

    return peaks, reaches, peaks_threshold


#########################################################################################
# re-read the data (that may have been modified)
kinect_t = kinect_mocap["time_stamps"]
kinect_data = kinect_mocap["time_series"]

WristLeft_Norm = get_joint_norm(kinect_mocap, "WristLeft")
WristRight_Norm = get_joint_norm(kinect_mocap, "WristRight")

WristLeft_Norm_f = lowpass_filter(kinect_t, WristLeft_Norm, cutoff=0.5)
WristRight_Norm_f = lowpass_filter(kinect_t, WristRight_Norm, cutoff=0.5)

peaks_left, reaches_left, thresh_left = get_reaches(
    kinect_t, WristLeft_Norm, WristLeft_Norm_f, title_txt="Left wrist"
)
peaks_right, reaches_right, thresh_right = get_reaches(
    kinect_t, WristRight_Norm, WristRight_Norm_f, title_txt="Right wrist"
)

if do_visualize:

    def plot_reaches(
        ax, t, wrist, wrist_f, i_peaks, reaches, label="wrist", color="b", thresh=None
    ):
        """ " Plot the reaches of the wrist"""
        ax.plot(t, wrist, ".", label=label, color=color)
        ax.plot(t, wrist_f, label=f"{label} filtered", color=color, alpha=0.2)

        # plot the threshold
        if thresh is not None:
            ax.axhline(
                y=thresh,
                color=color,
                linestyle="--",
                label=f"Threshold {label}",
            )
        for reach in reaches:
            # plot the start and end of the reach + a line
            ax.plot(
                reach["t_beg"],
                reach["beg_position"],
                "o",
                color="orange",
            )
            ax.plot(
                reach["t_end"],
                reach["end_position"],
                "o",
                color="r",
            )
            ax.plot(
                [reach["t_beg"], reach["t_end"]],
                [reach["beg_position"], reach["end_position"]],
                color="k",
                linestyle="--",
            )
            # plot the start and end of the reach on the filtered position
            ax.plot(
                reach["t_beg"],
                wrist_f[t == reach["t_beg"]],
                "x",
                color="orange",
            )
            ax.plot(
                reach["t_end"],
                wrist_f[t == reach["t_end"]],
                "x",
                color="r",
            )

        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Distance from the kinect (m)")
        ax.legend()

    # plot the signal and the peaks
    fig, ax = plt.subplots(figsize=(10, 5))
    if peaks_left is not None:
        plot_reaches(
            ax,
            kinect_t,
            WristLeft_Norm,
            WristLeft_Norm_f,
            peaks_left,
            reaches_left,
            label="Left wrist",
            color="b",
            thresh=thresh_left,
        )
    if peaks_right is not None:
        plot_reaches(
            ax,
            kinect_t,
            WristRight_Norm,
            WristRight_Norm_f,
            peaks_right,
            reaches_right,
            label="Right wrist",
            color="k",
            thresh=thresh_right,
        )

    plt.title("Reaches detected from the wrist position (low pass filtered @ 0.5Hz)")
    plt.show()

# Get the action zones from the raw kinect data

In [ ]:
def get_median_data_around_timestamp(data, time_stamps, t_stamp):
    """Get the median data around the timestamp t_stamp"""

    # find the index of the closest timestamp
    i_data = np.argmin(np.abs(time_stamps - t_stamp))

    # we want the median of the positions at i_data +/- 0.5 seconds
    # we assume that we have enough data before and after the index
    half_window = 15
    mask = range(i_data - half_window, i_data + half_window)
    data = data[mask]
    median_data = np.median(data, axis=0)

    # return median_data
    return {
        "median": median_data,
        "mask": mask,
    }


def get_median_xyz(kinect_mocap, reach_median, channel_name_X):
    """Get the x, y, z coordinates of a kinect channel in reach_median"""
    
    i_channel = get_kinect_channel_index(kinect_mocap, channel_name_X)
    # get the values of the channels of interest
    median_xyz = reach_median["median"][i_channel : i_channel + 3]
    return median_xyz

if reaches_left and reaches_right:
    print(f"first_reach_left_start time {reaches_left[0]["t_beg"]}")
    first_reach_left_start = get_median_data_around_timestamp(
        kinect_mocap["time_series"], 
        kinect_mocap["time_stamps"],
        reaches_left[0]["t_beg"]
    )

    print(f"first_reach_left_end time {reaches_left[0]["t_end"]}")
    first_reach_left_end = get_median_data_around_timestamp(
        kinect_mocap["time_series"], 
        kinect_mocap["time_stamps"],
        reaches_left[0]["t_end"]
    )

    print(f"first_reach_right_start time {reaches_right[0]["t_beg"]}")
    first_reach_right_start = get_median_data_around_timestamp(
        kinect_mocap["time_series"], 
        kinect_mocap["time_stamps"],
        reaches_right[0]["t_beg"]
    )
    print(f"first_reach_right_end time {reaches_right[0]["t_end"]}")
    first_reach_right_end = get_median_data_around_timestamp(
        kinect_mocap["time_series"], 
        kinect_mocap["time_stamps"],
        reaches_right[0]["t_end"]
    )


    first_reach_left_start_wrist_xyz = get_median_xyz(
        kinect_mocap, first_reach_left_start, "WristLeft_X"
    )
    first_reach_left_end_wrist_xyz = get_median_xyz(
        kinect_mocap, first_reach_left_end, "WristLeft_X"
    )

    first_reach_right_start_wrist_xyz = get_median_xyz(
        kinect_mocap, first_reach_right_start, "WristRight_X"
    )

    first_reach_right_end_wrist_xyz = get_median_xyz(
        kinect_mocap, first_reach_right_end, "WristRight_X"
    )


    print(
        f"Left wrist position at the beginning of the first reach: {first_reach_left_start_wrist_xyz}"
    )
    print(
        f"Left wrist position at the end of the first reach: {first_reach_left_end_wrist_xyz}"
    )
    print(
        f"Right wrist position at the beginning of the first reach: {first_reach_right_start_wrist_xyz}"
    )
    print(
        f"Right wrist position at the end of the first reach: {first_reach_right_end_wrist_xyz}"
    )

    first_reach_left_distance = np.linalg.norm(
        first_reach_left_end_wrist_xyz - first_reach_left_start_wrist_xyz
    )
    first_reach_right_distance = np.linalg.norm(
        first_reach_right_end_wrist_xyz - first_reach_right_start_wrist_xyz
    )
    print(f"Left wrist distance: {first_reach_left_distance:.2f} m")
    print(f"Right wrist distance: {first_reach_right_distance:.2f} m")

## Show the reaches and the all the switches from one hand to the other

In [ ]:
if reaches_left and reaches_right:

    def get_reach_list(reaches_left, reaches_right):
        """Make a single reach list from the left and right reaches, sorted by time"""
        reaches = []
        for reach in reaches_left:
            reach["wrist"] = "left"
            reaches.append(reach)
        for reach in reaches_right:
            reach["wrist"] = "right"
            reaches.append(reach)
        reaches = sorted(reaches, key=lambda x: x["t_beg"])

        return reaches

    # make a single reach list from the left and right

    reaches = get_reach_list(reaches_left, reaches_right)

    # print the reaches
    for i, reach in enumerate(reaches):
        print(
            f"{i:02d}: {reach['t_beg']:8.2f} -> {reach['t_end']:8.2f}, Duration: {reach['t_end'] - reach['t_beg']:5.2f}s, Wrist: {reach['wrist']}"
        )

    # find the switches between left and right
    def find_switches(reaches):
        """Find the switches between left and right reaches"""
        switches = []
        for i in range(len(reaches) - 1):
            if reaches[i]["wrist"] != reaches[i + 1]["wrist"]:
                switches.append(i)
        return switches

    switches = find_switches(reaches)
    # print the switches
    for i in switches:
        print(
            f"Switch at {i:02d}: {reaches[i]['t_beg']:8.2f} -> {reaches[i]['t_end']:8.2f}"
        )

    # the calibration reaches are the contiguous switches
    i_calibration_reaches = []
    i_calibration_reach_switch = np.where(np.diff(switches) == 1)[0]
    if len(i_calibration_reach_switch) == 0:
        print("No calibration reach found")
    else:
        i_calibration_reach_switch = i_calibration_reach_switch[0]
        i_calibration_reach = switches[i_calibration_reach_switch]

        # append the next index (as a switch implies two hands)
        i_calibration_reaches = np.append(i_calibration_reach, i_calibration_reach + 1)

    # print the reaches  identifying the calibration reaches
    for i, reach in enumerate(reaches):
        msg = ""
        if i in i_calibration_reaches:
            msg = ", Calibration reach"
            print(
                f"{i:02d}: {reach['t_beg']:8.2f} -> {reach['t_end']:8.2f}, Duration: {reach['t_end'] - reach['t_beg']:5.2f}s, Wrist: {reach['wrist'] }{msg}"
            )

    # plot the reaches with the switches
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, reach in enumerate(reaches):
        if reach["wrist"] == "left":
            color = "b"
        else:
            color = "k"
        ax.plot(
            [reach["t_beg"], reach["t_end"]],
            [reach["beg_position"], reach["end_position"]],
            color=color,
            linestyle="--",
        )
        ax.plot(
            reach["t_beg"],
            reach["beg_position"],
            "o",
            color=color,
        )
        ax.plot(
            reach["t_end"],
            reach["end_position"],
            "o",
            color=color,
        )
        # plot the switches
        if i in switches:
            ax.plot(
                reach["t_beg"],
                reach["beg_position"],
                "x",
                color="r",
                markersize=10,
                label="Switch",
            )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Distance from the kinect (m)")
    ax.legend()
    plt.show()

## Get the action zones from changes from one wrist to the other

Within one action zone, the participant is asked to perform a series of reaches with one hand (typically a block of 5 reaches).  
When the participant switches from one hand to the other, this is the end of the action zone with this hand. 

The next reach is performed with the other hand in the next action zone.

In [ ]:
# identify the blocks (i.e. the contiguous reaches in one action zone)

# in a block, the time between the reaches is less than 10s
reaches_starts = np.array([reach["t_beg"] for reach in reaches])
intervals = np.diff(reaches_starts)
in_block_time = intervals < 10  # classically 4-5 reaches in a 20s block, but minimum 2

# in a block, the hand is the same
reaches_hand = np.array([1 if reach["wrist"] == "right" else 0 for reach in reaches])
change_hand = np.diff(reaches_hand)
in_block_hand = change_hand == 0

# in a block, the hand is the same AND the time between reaches is less than 10s
in_block = in_block_time & in_block_hand

# get the changes in block
change_block = np.diff(in_block.astype(int))

# the next reach after a change_block is the start-end of a new block
start_blocks = np.where(change_block == 1)[0] + 1
end_blocks = np.where(change_block == -1)[0] + 1

# do not forget to add the first and last limits
if in_block[0]:
    start_blocks = np.insert(start_blocks, 0, 0)
if in_block[-1]:
    end_blocks = np.append(end_blocks, len(reaches) - 1)

for i in range(len(start_blocks)):
    print(f"Block {i:2d}: {start_blocks[i]:2d} -> {end_blocks[i]:2d}")

# identify the changes in hand from block to block
hand_blocks = np.array(
    [
        1 if reaches[start_blocks[i]]["wrist"] == "right" else 0
        for i in range(len(start_blocks))
    ]
)
change_hand_blocks = np.diff(hand_blocks)
in_condition = change_hand_blocks == 0
change_condition = np.diff(in_condition.astype(int))

block_change_hand = np.where(change_hand_blocks != 0)[0] + 1
block_start_condition = np.insert(block_change_hand, 0, 0)
block_end_condition = block_change_hand - 1
block_end_condition = np.append(block_end_condition, len(hand_blocks) - 1)

for i in range(len(block_start_condition)):
    r_start = start_blocks[block_start_condition[i]]
    r_end = end_blocks[block_end_condition[i]]
    print(
        f"Condition {i:2d}: block {block_start_condition[i]:2d} -> {block_end_condition[i]:2d} <=> reach {r_start:2d} -> {r_end:2d}"
    )

### plot the reaches and the blocs

In [ ]:
condition_names = ["sau-p", "sau-np", "mau-p", "mau-np"]
condition_colors = ["orange", "purple", "red", "green"]


fig, ax = plt.subplots(figsize=(10, 5))
for i, reach in enumerate(reaches):
    if reach["wrist"] == "left":
        color = "b"
    else:
        color = "k"
    ax.plot(
        [reach["t_beg"], reach["t_end"]],
        [reach["beg_position"], reach["end_position"]],
        color=color,
        linestyle="--",
    )
    ax.plot(
        reach["t_beg"],
        reach["beg_position"],
        "o",
        color=color,
    )
    ax.plot(
        reach["t_end"],
        reach["end_position"],
        "o",
        color=color,
    )

# plot the blocks
for i in range(len(start_blocks)):
    # make a grayed zone for the block
    ax.axvspan(
        reaches[start_blocks[i]]["t_beg"],
        reaches[end_blocks[i]]["t_end"],
        color="gray",
        alpha=0.2,
    )

# plot the conditions
for i in range(len(block_start_condition)):
    # make a grayed zone for the condition
    ax.axvspan(
        reaches[start_blocks[block_start_condition[i]]]["t_beg"],
        reaches[end_blocks[block_end_condition[i]]]["t_end"],
        color=condition_colors[i],
        alpha=0.2,
        label=condition_names[i],
    )

ax.set_xlabel("Time (s)")
ax.set_ylabel("Distance from the kinect (m)")
ax.set_title(
    "Actions zones and conditions from the reach sequence only (no markers needed)"
)
ax.legend()
plt.show()

# Guess a time correction if `time_correction` is nan

This is a workaround when no mouse data is available in csv files. 
The accuracy is within 1 second (or so), which is enough for the PANU analysis.

In [ ]:
# guess the time correction if time_correction is nan
# we guess the  correction from the first reach

if mouse_markers and kinect_mocap:
    markers_t = mouse_markers["time_stamps"]
    kinect_t = kinect_mocap["time_stamps"]

    print("kinect_t[0]:", kinect_t[0])

    # first action start time
    print("markers_t[0]:", markers_t[0])

    # first sau reach time
    i_reach_after_cal_sau_switch = switches[1] + 1
    first_sau_reach = reaches[i_reach_after_cal_sau_switch]["t_beg"]
    print("first_sau_reach:", first_sau_reach)

    # guess the time correction

    correction = markers_t[0] - (first_sau_reach - 1)

    print("guess time correction:", correction)

    print_items(reach)


if np.isnan(time_correction):
    kinect_mocap = get_stream(xdf_data, "MoCap", "EuroMov-Mocap-Kinect")
    kinect_markers = get_stream(xdf_data, "Markers", "EuroMov-Markers-Kinect")
    if kinect_mocap:
        kinect_mocap["time_stamps"] = kinect_mocap["time_stamps"] + correction
    if kinect_markers:
        kinect_markers["time_stamps"] = kinect_markers["time_stamps"] + correction

    for reach in reaches:
        reach["t_beg"] = reach["t_beg"] + correction
        reach["t_end"] = reach["t_end"] + correction
        # NOTE: the index is unchanged

# Get the start and stop of the sau-mau-cal conditions from the markers


In [ ]:
def get_start_stop_times_markers():
    if mouse_markers:
        start_times_from_mouse_markers, stop_times_from_mouse_markers = (
            get_actions_start_stop_times_from_mouse_markers(mouse_markers)
        )
        # print("Mouse markers:")
        print_actions_start_stop_times(
            (start_times_from_mouse_markers, stop_times_from_mouse_markers)
        )
        return start_times_from_mouse_markers, stop_times_from_mouse_markers

    if event_to_nic_markers:
        start_times_from_event_ide_tonic, stop_times_from_event_ide_tonic = (
            get_actions_start_stop_times_from_event_ide_TONIC(event_to_nic_markers)
        )
        # print("Event IDE to NIC markers:")
        print_actions_start_stop_times(
            (start_times_from_event_ide_tonic, stop_times_from_event_ide_tonic)
        )
        return start_times_from_event_ide_tonic, stop_times_from_event_ide_tonic

    return None, None


actions_start_times, actions_stop_times = get_start_stop_times_markers()

print_actions_start_stop_times((actions_start_times, actions_stop_times))

# plot the right and left wrist distances with the  start and stop times of the recording blocks
fig, ax = plt.subplots(figsize=(10, 5))
# plot the left and right wrist norms
ax.plot(
    kinect_mocap["time_stamps"],
    WristLeft_Norm,
    ".",
    label="Left wrist ",
    color="b",
)
ax.plot(
    kinect_mocap["time_stamps"],
    WristRight_Norm,
    ".",
    label="Right wrist ",
    color="k",
)

if actions_start_times is not None and actions_stop_times is not None:
    for i, (start, stop) in enumerate(zip(actions_start_times, actions_stop_times)):
        ax.vlines(
            start,
            ymin=np.min(WristLeft_Norm),
            ymax=np.max(WristLeft_Norm),
            color="g",
            linestyle="--",
        )
        ax.vlines(
            stop,
            ymin=np.min(WristLeft_Norm),
            ymax=np.max(WristLeft_Norm),
            color="r",
            linestyle="--",
        )

ax.set_xlabel("Time (s)")
ax.set_ylabel("Distance from the kinect (m)")
ax.set_title(f"{xdf_fullFname} \n actions start/stop times from NIC markers")
ax.legend()
plt.show()

# Get the reaches that are in the action zones 

In [ ]:
# get the paretic side from the reach sequence
# NOTE: the first reach is from the paretic side
if reaches:
    if reaches[0]["wrist"] == "left":
        paretic = "left"
    else:
        paretic = "right"
    print(f"Paretic side: {paretic}")

# get the action reaches etc.
# action means that the reach is within the start-stop block time limits
if actions_start_times is not None and actions_stop_times is not None:
    is_action = np.array([False] * len(actions_start_times))
    is_paretic = np.array([False] * len(actions_start_times))
    nb_reaches_p = np.array([0] * len(actions_start_times))
    nb_reaches_np = np.array([0] * len(actions_start_times))
    labels = np.array(["unknown"] * len(actions_start_times), dtype="<U15")
    # for each start-stop pair, count the number of reaches in the action block
    for i, (start, stop) in enumerate(zip(actions_start_times, actions_stop_times)):
        # TODO: this should not be needed if we better compute the start and end times
        start = start - 0.5  # wrist filtered at 0.5 Hz => start is too early
        for reach in reaches:
            if reach["t_beg"] >= start and reach["t_end"] <= stop:  # inside the block
                if reach["wrist"] == paretic:
                    nb_reaches_p[i] += 1
                else:
                    nb_reaches_np[i] += 1

        if nb_reaches_p[i] >= 2:
            is_action[i] = True
            is_paretic[i] = True
            labels[i] = "Paretic"

        if nb_reaches_np[i] >= 2:
            is_action[i] = True
            is_paretic[i] = False
            labels[i] = "Non-paretic"

    # get the action zones
    action_start_times = actions_start_times[is_action]
    action_stop_times = actions_stop_times[is_action]
    action_nb_reaches_p = nb_reaches_p[is_action]
    action_nb_reaches_np = nb_reaches_np[is_action]
    action_is_paretic = is_paretic[is_action]
    action_labels = labels[is_action]

    # print the action zones
    print("Action start and stop times:")
    for i, (start, stop, lab) in enumerate(
        zip(action_start_times, action_stop_times, action_labels)
    ):
        print(
            f"{i:02d}: {start:8.2f} -> {stop:8.2f}, Duration: {stop - start:5.2f}s, {lab}"
        )

    # plot the action zones
    fig, ax = plt.subplots(figsize=(10, 5))
    # plot the left and right wrist norms
    ax.plot(
        kinect_mocap["time_stamps"],
        WristLeft_Norm,
        ".",
        label=f"Left wrist ({'Paretic' if 'left' in paretic else 'Non-paretic'})",
        color="b",
    )
    ax.plot(
        kinect_mocap["time_stamps"],
        WristRight_Norm,
        ".",
        label=f"Right wrist ({'Paretic' if 'right' in paretic else 'Non-paretic'})",
        color="k",
    )
    for i, (start, stop) in enumerate(zip(action_start_times, action_stop_times)):
        ax.vlines(
            start,
            ymin=np.min(WristLeft_Norm),
            ymax=np.max(WristLeft_Norm),
            color="g",
            linestyle="--",
        )
        ax.vlines(
            stop,
            ymin=np.min(WristLeft_Norm),
            ymax=np.max(WristLeft_Norm),
            color="r",
            linestyle="--",
        )
        # add a label to the zone
        y_pos = np.min(WristLeft_Norm) - 0.1
        ax.text(
            (start + stop) / 2,
            y_pos + 0.05,  # np.min(left_wrist_distance),
            f"{action_nb_reaches_np[i]:02d}",
            ha="center",
            va="bottom",
            color="k",
            fontsize=10,
        )
        ax.text(
            (start + stop) / 2,
            y_pos + 0.04,  # np.min(left_wrist_distance),
            f"{action_nb_reaches_p[i]:02d}",
            ha="center",
            va="top",
            color="blue",
            fontsize=10,
        )

        ylim = ax.get_ylim()
        ax.set_ylim(y_pos, ylim[1])
        ax.legend()
        plt.title(
            f"{xdf_fullFname} \n actions start/stop times from NIC markers : {len(action_start_times)} actions with reaches counted"
        )
        plt.show()

# Define the the sau-mau-cal conditions

In [ ]:
def set_sau_mau_zones(action_labels):
    """Set the SAU and MAU zones from the action labels"""

    sau = {
        "name": "SAU",
        "start time": action_start_times[0],
        "end time": -1,
        "nb Paretic": 0,
        "nb Non-paretic": 0,
    }
    mau = {
        "name": "MAU",
        "start time": -1,
        "end time": action_stop_times[-1],
        "nb Paretic": 0,
        "nb Non-paretic": 0,
    }
    # when the bloc label changes from Non-paretic to Paretic, this is the end of sau
    # as the sequence : sau-p, sau-np, mau-p, mau-np
    i_end_sau = []  # in case we have multiple changes...
    for i in range(len(action_labels) - 1):
        if action_labels[i] != action_labels[i + 1]:
            if action_labels[i] == "Non-paretic":
                i_end_sau.append(i)

    # if we have more than 1 i_end_sau, take the last one (experimental problem in the previous trials...)
    i_end_SAU = i_end_sau[-1]

    sau["end time"] = action_stop_times[i_end_SAU]
    mau["start time"] = action_start_times[i_end_SAU + 1]

    # add 0.5 seconds to the start and end time of sau and mau
    # Reason: lp filter tends to shift the start time of the reach (see reach figures)
    # TODO: this should not be needed if we better compute the start and end times
    # that is, compute the start and end times from the distance to the TARGET
    sau["start time"] -= 0.5
    sau["end time"] += 0.5
    mau["start time"] -= 0.5
    mau["end time"] += 0.5

    # add the number of labels in the sau and mau
    for i in range(len(action_labels)):
        if i <= i_end_SAU:
            if action_labels[i] == "Non-paretic":
                sau["nb Non-paretic"] += 1
            else:
                sau["nb Paretic"] += 1
        else:
            if action_labels[i] == "Non-paretic":
                mau["nb Non-paretic"] += 1
            else:
                mau["nb Paretic"] += 1

    # add the duration of the sau and mau
    sau["duration"] = sau["end time"] - sau["start time"]
    mau["duration"] = mau["end time"] - mau["start time"]

    return sau, mau


def print_zone_dict(d):
    """Print a sau-mau-cal dictionary"""
    print(
        f"{d['name']:5s}: {d['start time']:5.2f} -> {d['end time']:.2f}: {d['duration']:7.2f}s, Blocks: {d['nb Paretic']} Paretic, {d['nb Non-paretic']} Non-paretic"
    )


####################################################################################
sau, mau = set_sau_mau_zones(action_labels)


# add labels to the reaches
for reach in reaches:
    # add the sau and mau labels to the reaches
    if reach["t_beg"] >= sau["start time"] and reach["t_end"] <= sau["end time"]:
        reach["condition"] = "sau"
    elif reach["t_beg"] >= mau["start time"] and reach["t_end"] <= mau["end time"]:
        reach["condition"] = "mau"
    else:
        reach["condition"] = "unknown"
    # add the paretic label to the reaches
    if reach["wrist"] == paretic:
        reach["arm"] = "Paretic"
    else:
        reach["arm"] = "Non-paretic"


reaches_panu = [reach for reach in reaches if reach["condition"] != "unknown"]
reaches_unknown = [reach for reach in reaches if reach["condition"] == "unknown"]


# NOTE: unknown should be the calibration reaches
# # We may have multiple calibration reaches: we take the first one


def first_calibration_reach_pair(reaches_unknown, start_time=None):
    """Get the calibration reach couple from the unknown reaches"""
    # we look for the first two left+right reaches that are less than 30 seconds apart
    cal_start_times = np.array([reach["t_beg"] for reach in reaches_unknown])
    cal_wrists = np.array([reach["wrist"] for reach in reaches_unknown])

    if start_time is None:
        start_time = reaches_unknown[0]["t_beg"]

    # remove the reaches that are before the start time
    cal_start_times = cal_start_times[cal_start_times >= start_time]
    cal_wrists = cal_wrists[cal_start_times >= start_time]

    # retrieve the first left+right pair less than 30 seconds apart
    d_cal_start_times = np.diff(cal_start_times)
    for i in range(len(d_cal_start_times)):
        if d_cal_start_times[i] < 30:
            if cal_wrists[i] != cal_wrists[i + 1]:
                return (reaches_unknown[i], reaches_unknown[i + 1])
    return None, None


def get_calibrations(reaches_unknown):
    """Get the calibration reaches from the unknown reaches"""
    # we look for the first two left+right reaches that are less than 30 seconds apart
    # we may have multiple calibration reaches: we take the first pair
    start_time = reaches_unknown[0]["t_beg"]
    cals = []

    while len(cals) < 1 and start_time < reaches_unknown[-1]["t_end"]:
        # get the first left+right pair less than 30 seconds apart
        cal_reach_1, cal_reach_2 = first_calibration_reach_pair(
            reaches_unknown, start_time
        )
        if cal_reach_1 is not None and cal_reach_2 is not None:
            cals.append(
                {
                    "name": f"CAL_{len(cals)}",
                    "start time": cal_reach_1["t_beg"],
                    "end time": cal_reach_2["t_end"],
                    "duration": cal_reach_2["t_end"] - cal_reach_1["t_beg"],
                    "nb Non-paretic": 1,
                    "nb Paretic": 1,
                }
            )
            start_time = cal_reach_2["t_end"] + 1  # add 1 second to the start time

    return cals


calibrations = get_calibrations(reaches_unknown)


cal_start_times = np.array([reach["t_beg"] for reach in reaches_unknown])
d_cal_start_times = np.diff(cal_start_times)

# print the calibration zone(s)
for cal in calibrations:
    print_zone_dict(cal)

# print the sau and mau zones
print_zone_dict(sau)
print_zone_dict(mau)

# Get the reaches for the calibration 

In [ ]:
def get_calib_xyz(kinect_mocap, reach):
    """Get the calibration xyz of the wrist at the beginning and end of the reach"""

    joint_name = f"Wrist{'Left' if reach['wrist'] == 'left' else 'Right'}"

    median_arround_reach_beg = get_median_data_around_timestamp(
        kinect_mocap["time_series"], kinect_mocap["time_stamps"], reach["t_beg"]
    )
    calib_xyz_at_beg = get_median_xyz(
        kinect_mocap, median_arround_reach_beg, f"{joint_name}_X"
    )

    median_arround_reach_end = get_median_data_around_timestamp(
        kinect_mocap["time_series"], kinect_mocap["time_stamps"], reach["t_end"]
    )
    calib_xyz_at_end = get_median_xyz(
        kinect_mocap, median_arround_reach_end, f"{joint_name}_X"
    )
    # get the median of the two positions

    return {
        "xyz_beg": calib_xyz_at_beg,
        "xyz_end": calib_xyz_at_end,
        "distance": np.linalg.norm(calib_xyz_at_end - calib_xyz_at_beg),
        "t_beg": reach["t_beg"],
        "t_end": reach["t_end"],
        "joint": joint_name,
    }


# get the reaches in the cal zone
reaches_cal = [reach for reach in reaches if reach["condition"] == "unknown"]

print(f"Number of reaches in the calibration zone: {len(reaches_cal)}")
# print the reaches in the calibration zone
for i, reach in enumerate(reaches_cal):
    print(
        f"{i:02d}: {reach['t_beg']:8.2f} -> {reach['t_end']:8.2f}, Duration: {reach['t_end'] - reach['t_beg']:5.2f}s, Wrist: {reach['wrist']}, Condition: {reach['condition']}, Arm: {reach['arm']}"
    )

# left wrist calibration
calib_xyz_0 = get_calib_xyz(kinect_mocap, reaches_cal[0])
print(f"{calib_xyz_0['joint']}")
print(f"@beg: {calib_xyz_0['xyz_beg']}")
print(f"@end: {calib_xyz_0['xyz_end']}")
print(f"dist: {calib_xyz_0['distance']:.4f} m")

# right wrist calibration
calib_xyz_1 = get_calib_xyz(kinect_mocap, reaches_cal[1])
print(f"{calib_xyz_1['joint']}")
print(f"@beg: {calib_xyz_1['xyz_beg']}")
print(f"@end: {calib_xyz_1['xyz_end']}")
print(f"dist: {calib_xyz_1['distance']:.4f} m")

# plot the calibration reaches
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    kinect_mocap["time_stamps"],
    WristLeft_Norm,
    ".",
    label=f"Left wrist ({'Paretic' if 'left' in paretic else 'Non-paretic'})",
    color="b",
)
ax.plot(
    kinect_mocap["time_stamps"],
    WristRight_Norm,
    ".",
    label=f"Right wrist ({'Paretic' if 'right' in paretic else 'Non-paretic'})",
    color="k",
)
for i, reach in enumerate(reaches_cal):
    if reach["wrist"] == "left":
        color = "b"
    else:
        color = "k"
    ax.plot(
        [reach["t_beg"], reach["t_end"]],
        [reach["beg_position"], reach["end_position"]],
        color=color,
        linestyle="--",
    )
    ax.plot(
        reach["t_beg"],
        reach["beg_position"],
        "o",
        color="g",
    )
    ax.plot(
        reach["t_end"],
        reach["end_position"],
        "o",
        color="r",
    )

ax.set_xlabel("Time (s)")
ax.set_ylabel("Distance from the kinect (m)")
ax.legend()
ax.set_title(f"{xdf_fullFname} \n Calibration reaches: {len(reaches_cal)} identified")
plt.show()

## Get the target position

In [ ]:
# first reach positions
target_xyz = np.mean(
    [first_reach_left_end_wrist_xyz, first_reach_right_end_wrist_xyz], axis=0
)
print(f"Left wrist first position: {first_reach_left_end_wrist_xyz}")
print(f"Right wrist first position: {first_reach_right_end_wrist_xyz}")

calib_reach_0 = [
    reach for reach in reaches if reach["t_beg"] == calibrations[0]["start time"]
]
calib_xyz_0 = get_calib_xyz(kinect_mocap, calib_reach_0[0])
calib_reach_1 = [
    reach for reach in reaches if reach["t_end"] == calibrations[0]["end time"]
]
calib_xyz_1 = get_calib_xyz(kinect_mocap, calib_reach_1[0])


# Calibration positions
if calib_xyz_0["joint"] == "WristLeft":
    calib_xyz_left = calib_xyz_0["xyz_end"]
    calib_xyz_right = calib_xyz_1["xyz_end"]
else:
    calib_xyz_left = calib_xyz_1["xyz_end"]
    calib_xyz_right = calib_xyz_0["xyz_end"]


print(f"Left wrist calibration position: {calib_xyz_left}")
print(f"Right wrist calibration position: {calib_xyz_right}")
target_xyz = np.mean([calib_xyz_0["xyz_end"], calib_xyz_1["xyz_end"]], axis=0)


# Format the arrays for better readability
txt = ""
for x in target_xyz:
    txt += f"{x:.3f} "
print(f"target_xyz position (m): {txt}")

## Compute the distance to the target, for both wrists and both shoulders

In [ ]:
def get_distance_to_target(xyz, target_xyz):
    """Get the distance to the target"""

    # check that the target is a 1D array of 3 elements
    if target_xyz.ndim != 1 or target_xyz.shape[0] != 3:
        raise ValueError("target_xyz must be a 1D array of 3 elements")

    # check that xyz is either a 1D or 2D array of 3 elements by row
    if xyz.ndim == 1:
        # reshape it to a 2D array (with one row)
        xyz = np.array([xyz])
    elif xyz.ndim == 2:
        # check that it has 3 columns
        if xyz.shape[1] != 3:
            raise ValueError("xyz must be a 2D array of 3 elements")
    else:
        raise ValueError("xyz must be a 1D or 2D array")

    # get the distance to the target
    distance_to_target = np.linalg.norm(xyz - target_xyz, axis=1)

    return distance_to_target


def get_kinect_channel_xyz(kinect_mocap, channel_name_X):
    """Get the x, y, z coordinates of a kinect channel in the kinect_mocap stream"""
    i_channel = get_kinect_channel_index(kinect_mocap, channel_name_X)
    channel_data = kinect_mocap["time_series"][:, i_channel : i_channel + 3]
    return channel_data


left_wrist_xyz = get_kinect_channel_xyz(kinect_mocap, "WristLeft_X")
right_wrist_xyz = get_kinect_channel_xyz(kinect_mocap, "WristRight_X")
left_shoulder_xyz = get_kinect_channel_xyz(kinect_mocap, "ShoulderLeft_X")
right_shoulder_xyz = get_kinect_channel_xyz(kinect_mocap, "ShoulderRight_X")

left_wrist_distance = get_distance_to_target(left_wrist_xyz, target_xyz)
right_wrist_distance = get_distance_to_target(right_wrist_xyz, target_xyz)
left_shoulder_distance = get_distance_to_target(left_shoulder_xyz, target_xyz)
right_shoulder_distance = get_distance_to_target(right_shoulder_xyz, target_xyz)


print(left_wrist_distance.shape)

# plot the left_wrist_distance over time
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    kinect_mocap["time_stamps"],
    left_wrist_distance,
    ".",
    label=f"Left wrist distance({'Paretic' if 'left' in paretic else 'Non-paretic'})",
    color="b",
)
ax.plot(
    kinect_mocap["time_stamps"],
    right_wrist_distance,
    ".",
    label=f"Right wrist distance({'Paretic' if 'right' in paretic else 'Non-paretic'})",
    color="k",
)
# plot the left and right shoulder distances
ax.plot(
    kinect_mocap["time_stamps"],
    left_shoulder_distance,
    ".",
    label="Left shoulder distance",
    markerfacecolor="w",
    markeredgecolor="b",
    markeredgewidth=0.5,
)
ax.plot(
    kinect_mocap["time_stamps"],
    right_shoulder_distance,
    ".",
    label="Right shoulder distance",
    markerfacecolor="w",
    markeredgecolor="k",
    markeredgewidth=0.5,
)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Distance to target (m)")
ax.legend()
plt.show()

# Re-compute the reaches start and stop times from the distance to the target data

In [ ]:
def get_reach_end_from_distance_to_target(reach, kinect_time, distance_to_target):
    """Get the reach final position from the distance to target"""

    # define a window where to find the minimum distance to target
    i_beg = reach["t_beg_i"]
    i_end = reach["t_end_i"]
    half_window = (i_end - i_beg) // 2
    i_window = range(i_beg - half_window, i_end + half_window)
    i_reach = np.argmin(distance_to_target[i_window]) + i_beg - half_window - 1

    # get the reach end median value + mask
    median_distance_to_target_at_end = get_median_data_around_timestamp(
        distance_to_target,
        kinect_time,
        kinect_time[i_reach],
    )

    return {
        "i_window": i_window,
        "median_distance": median_distance_to_target_at_end["median"],
        "median_mask": median_distance_to_target_at_end["mask"],
        "timestamp": kinect_time[i_reach],
        "i_timestamp": i_reach,
        "distance": distance_to_target[i_reach],
    }


def get_reach_from_filtered_distance_to_target(
    reach, kinect_time, left_distance, right_distance
):
    """Get the reach start and end positions from the filtered distance to target"""

    # if the reach is left, use the left distance
    if reach["wrist"] == "left":
        distance_to_target = left_distance
    else:
        distance_to_target = right_distance

    # low pass filter the distance to target at 2.5 Hz
    # NOTE: this "optimal" cutoff was determined by Faity http://doi.org/10.3390/s22072735
    filtered_distance = lowpass_filter(
        kinect_time,  # kinect_mocap["time_stamps"],
        distance_to_target,
        cutoff=2.5,
    )

    # get the derivative of the filtered distance to target
    filtered_velocity = np.gradient(filtered_distance, kinect_time)

    # Step 1: Get the reach final position around the guessed reach END (reach["t_end"])
    # We scout for the  minimum distance to target between two velocity peaks:
    #  - from the negative velocity peak before the guessed END
    #  - to the positive velocity peak after the guessed END
    # Step 2: Get the reach start position
    # We scout for the last negative velocity before the negative velocity peak before the guessed END

    def get_negative_peak_index(signal, i_beg, i_end):
        """Get the index of the negative peak in a signal between two indices"""
        i_min_vel = np.argmin(signal[i_beg:i_end]) + i_beg
        return i_min_vel

    # 1.1. get the index of the negative velocity peak before the guessed END
    i_neg_vel_peak_before = get_negative_peak_index(
        filtered_velocity,
        reach["t_beg_i"],
        reach["t_end_i"],
    )

    # 1.2. get the index of the positive velocity peak after the guessed END
    window_size = 60  # 2 s = typical inter-reach time
    i_pos_vel_peak_after = get_negative_peak_index(
        -filtered_velocity,  # invert the sign to find the positive peak
        reach["t_end_i"],
        reach["t_end_i"] + window_size,
    )
    # As we have the two velocity peaks, we can get the reach end position...
    i_end_reach = get_negative_peak_index(
        filtered_distance,
        i_neg_vel_peak_before,
        i_pos_vel_peak_after,
    )

    # ... and the reach start position
    i_start_reach = index_of_last_negative_velocity_before_peak(
        kinect_time, filtered_velocity, t_end_i=i_neg_vel_peak_before
    )

    to_return = {
        "i_start_reach": i_start_reach,
        "i_end_reach": i_end_reach,
        "filtered_distance": filtered_distance,
        "filtered_velocity": filtered_velocity,
    }

    do_plot_debug = False
    if do_plot_debug:
        fig, ax = plt.subplots(figsize=(10, 5))

        ax.plot(
            kinect_time,  # kinect_mocap["time_stamps"],
            distance_to_target,
            ".",
            label="Raw distance to target",
            color="k",
        )

        ax.plot(
            kinect_time,  # kinect_mocap["time_stamps"],
            filtered_distance,
            ".-",
            linewidth=0.5,
            label="Filtered distance to target",
            color="b",
        )
        ax.plot(
            kinect_time,  # kinect_mocap["time_stamps"],
            filtered_velocity,
            ".-",
            linewidth=0.25,
            markersize=0.2,
            label="Derivative of distance to target",
            color="r",
        )
        # plot the reach start
        ax.plot(
            kinect_time,  # kinect_mocap["time_stamps"][i_start_reach],
            filtered_distance[i_start_reach],
            "o",
            label="new Reach start",
            color="g",
        )
        ax.axvline(
            kinect_time,  # kinect_mocap["time_stamps"][i_start_reach],
            color="g",
            linestyle="--",
        )
        # plot the reach end
        ax.plot(
            kinect_time[i_end_reach],  # kinect_mocap["time_stamps"][i_end_reach],
            filtered_distance[i_end_reach],
            "o",
            label="new Reach end",
            color="r",
        )
        ax.axvline(
            kinect_time[i_end_reach],  # kinect_mocap["time_stamps"][i_end_reach],
            color="r",
            linestyle="--",
        )
        # vertical grey line at i_beg and i_end
        ax.axvline(
            kinect_time[reach["t_beg_i"]],
            color="grey",
            linestyle="--",
            label="OLD Reach start",
        )
        ax.axvline(
            kinect_time[reach["t_end_i"]],
            color="grey",
            linestyle="--",
        )

        # vertical lines
        ax.axvline(
            kinect_time[i_neg_vel_peak_before],
            color="c",
            linestyle="-",
            label="Negative velocity peak before",
        )

        ax.axvline(
            kinect_time[i_pos_vel_peak_after],
            color="r",
            linestyle="-",
            label="Positive velocity peak after",
        )

        ax.axvline(
            kinect_time[reach["t_end_i"] + window_size],
            color="g",
            linestyle="-",
            label="end of search window  for positive peak",
        )

        # horizontal line at 0
        ax.axhline(0, color="k", linestyle="--")

        # set the x limits to the reach time
        ax.set_xlim(reach["t_beg"] - 10, reach["t_end"] + 10)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Distance to target (m)")
        ax.legend()
        plt.show()

    return to_return


#######################################################################################
if kinect_mocap:
    for reach in reaches[0:2]:
        reach_limits = get_reach_from_filtered_distance_to_target(
            reach,
            kinect_mocap["time_stamps"],
            left_wrist_distance,
            right_wrist_distance,
        )

    print("--- reach")
    print_items(reach)
    print("--- reach_limits")
    print_items(reach_limits)
    print("---")

## Plot the reaches obtained from the distance to the target data

In [ ]:
def plot_one_reach_two_axes_new_limits(reach, new_reach_limits, distance, title_txt):
    """Plot the reach with the new limits computed from the filtered distance to target"""

    t = kinect_mocap["time_stamps"]

    start_of_reach_i = new_reach_limits["i_start_reach"]
    end_of_reach_i = new_reach_limits["i_end_reach"]
    filtered_distance = new_reach_limits["filtered_distance"]
    filtered_velocity = new_reach_limits["filtered_velocity"]

    t_beg_i = reach["t_beg_i"]
    t_end_i = reach["t_end_i"]

    def plot_one_sub(ax):
        # plot the distance to target
        ax.plot(
            t,
            distance,
            ".",
            label="Distance (raw)",
            color="b",
            # linewidth=0.2,
            # markersize=0.5,
        )

        # plot the filtered distance to target
        ax.plot(
            t,
            filtered_distance,
            ".-",
            label="Filtered distance to target",
            color="b",
            linewidth=0.2,
            markersize=0.5,
        )

        # plot the OLD reach start and end
        ax.vlines(
            t[t_beg_i],
            ymin=np.min(distance),
            ymax=np.max(distance),
            color="g",
            linestyle="--",
            alpha=0.2,
            label="Beg reach",
        )
        ax.vlines(
            t[t_end_i],
            ymin=np.min(distance),
            ymax=np.max(distance),
            color="r",
            alpha=0.2,
            linestyle="--",
            label="End reach",
        )
        # plot the NEW reach start and end
        ax.vlines(
            t[start_of_reach_i],
            ymin=np.min(distance),
            ymax=np.max(distance),
            color="green",
            linestyle="--",
            label="Beg reach (new)",
        )
        ax.vlines(
            t[end_of_reach_i],
            ymin=np.min(distance),
            ymax=np.max(distance),
            color="red",
            linestyle="--",
            label="End reach (new)",
        )
        # plot the NEW reach start and end values 
        ax.plot(
            t[start_of_reach_i],
            filtered_distance[start_of_reach_i],
            "*",
            color="green",
            markersize=10,
        )
        ax.plot(
            t[end_of_reach_i],
            filtered_distance[end_of_reach_i],
            "*",
            color="red",
            markersize=10,
        )


        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Distance to the target (m)")

    # plot the data with 2 subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5))

    plot_one_sub(ax1)
    ax1.set_title(title_txt)
    plot_one_sub(ax2)
    t_zoom = t[start_of_reach_i] - 2, t[end_of_reach_i] + 2
    ax2.set_xlim(t_zoom)
    # ax2.set_title("Zoom on the reach of interest")
    from matplotlib.patches import ConnectionPatch

    xy1 = (float(t[end_of_reach_i]), float(np.min(distance)))
    xy2 = (float(t[end_of_reach_i]), float(np.max(distance)))
    connection_end = ConnectionPatch(
        xyA=xy1,
        xyB=xy2,
        coordsA="data",
        coordsB="data",
        axesA=ax1,
        axesB=ax2,
        color="red",
    )
    ax2.add_artist(connection_end)

    xy1 = (t[start_of_reach_i], np.min(distance))
    xy2 = (t[start_of_reach_i], np.max(distance))
    connection_beg = ConnectionPatch(
        xyA=xy1,
        xyB=xy2,
        coordsA="data",
        coordsB="data",
        axesA=ax1,
        axesB=ax2,
        color="green",
    )
    ax2.add_artist(connection_beg)

    plt.show()


##########################################################################################################
# TODO: make a single window with a button to switch between reaches (eg matplotib button)
reaches_to_plot = reaches[0:2] # only the first two reaches (for visualization)
for reach, i in zip(reaches_to_plot, range(len(reaches_to_plot))):
    reach_limits = get_reach_from_filtered_distance_to_target(
        reach,
        kinect_mocap["time_stamps"],
        left_wrist_distance,
        right_wrist_distance,
    )

    # if the reach is left, use the left distance
    if reach["wrist"] == "left":
        distance_to_target = left_wrist_distance
    else:
        distance_to_target = right_wrist_distance
    plot_one_reach_two_axes_new_limits(
        reach,
        reach_limits,
        distance_to_target,
        f"Reach {i} -- {reach["wrist"]}",
    )

reach_ = reach.copy()

In [ ]:
reach = reach_.copy()


def add_new_limits(reach, reach_limits):
    # change the reach using the new limits
    i_beg_new = reach_limits["i_start_reach"]
    i_end_new = reach_limits["i_end_reach"]
    reach["i_beg_distance"] = i_beg_new
    reach["i_end_distance"] = i_end_new
    reach["t_beg_distance"] = kinect_mocap["time_stamps"][i_beg_new]
    reach["t_end_distance"] = kinect_mocap["time_stamps"][i_end_new]
    reach["beg_distance"] = reach_limits["filtered_distance"][i_beg_new]
    reach["end_distance"] = reach_limits["filtered_distance"][i_end_new]
    reach["filtered_distance"] = reach_limits["filtered_distance"]
    reach["filtered_velocity"] = reach_limits["filtered_velocity"]
    return reach


print_items(reach_limits)

print_items(reach)
reach = add_new_limits(reach, reach_limits)
print("--- @")
print_items(reach)

In [ ]:
for reach in reaches:
    reach_limits = get_reach_from_filtered_distance_to_target(
        reach,
        kinect_mocap["time_stamps"],
        left_wrist_distance,
        right_wrist_distance,
    )
    reach = add_new_limits(reach, reach_limits)

# Add the PANU ingredients to the reaches


In [ ]:
for reach in reaches:
    # if it is a left reach, add the left wrist distance
    if reach["wrist"] == "left":
        reach["wrist_start_distance"] = get_median_data_around_timestamp(
            left_wrist_distance, kinect_mocap["time_stamps"], reach["t_beg"]
        )["median"]
        reach["wrist_end_distance"] = get_median_data_around_timestamp(
            left_wrist_distance, kinect_mocap["time_stamps"], reach["t_end"]
        )["median"]
        # add the shoulder distance
        reach["shoulder_start_distance"] = get_median_data_around_timestamp(
            left_shoulder_distance, kinect_mocap["time_stamps"], reach["t_beg"]
        )["median"]
        reach["shoulder_end_distance"] = get_median_data_around_timestamp(
            left_shoulder_distance, kinect_mocap["time_stamps"], reach["t_end"]
        )["median"]
    # if it is a right reach, add the right wrist distance
    if reach["wrist"] == "right":
        reach["wrist_start_distance"] = get_median_data_around_timestamp(
            right_wrist_distance, kinect_mocap["time_stamps"], reach["t_beg"]
        )["median"]
        reach["wrist_end_distance"] = get_median_data_around_timestamp(
            right_wrist_distance, kinect_mocap["time_stamps"], reach["t_end"]
        )["median"]
        # add the shoulder distance
        reach["shoulder_start_distance"] = get_median_data_around_timestamp(
            right_shoulder_distance, kinect_mocap["time_stamps"], reach["t_beg"]
        )["median"]
        reach["shoulder_end_distance"] = get_median_data_around_timestamp(
            right_shoulder_distance, kinect_mocap["time_stamps"], reach["t_end"]
        )["median"]

    # for all the reaches, add the traveled distances
    reach["shoulder_travel"] = (
        reach["shoulder_end_distance"] - reach["shoulder_start_distance"]
    )
    reach["wrist_travel"] = reach["wrist_end_distance"] - reach["wrist_start_distance"]
    reach["S/W ratio"] = reach["shoulder_travel"] / reach["wrist_travel"]


def add_panu_ingredientts(reach):
    """Add the panu ingredients to the reach"""

    # select the FILTERED distance
    wrist_distance = reach["filtered_distance"]

    # select the shoulder distance according to the wrist
    shoulder_distance = right_shoulder_distance
    if reach["wrist"] == "left":
        shoulder_distance = left_shoulder_distance

    # add the shoulder distance
    reach["shoulder_start_new"] = shoulder_distance[reach["i_beg_distance"]]
    reach["shoulder_end_new"] = shoulder_distance[reach["i_end_distance"]]

    # add the wrist distance
    reach["wrist_start_new"] = wrist_distance[reach["i_beg_distance"]]
    reach["wrist_end_new"] = wrist_distance[reach["i_end_distance"]]

    # add the traveled distances
    reach["shoulder_travel_new"] = (
        reach["shoulder_end_new"] - reach["shoulder_start_new"]
    )
    reach["wrist_travel_new"] = reach["wrist_end_new"] - reach["wrist_start_new"]
    reach["S/W ratio new"] = reach["shoulder_travel_new"] / reach["wrist_travel_new"]

    return reach


# print the content of the reaches
for i, reach in enumerate(reaches):
    reach = add_panu_ingredientts(reach)
    txt1 = f"{i:02d}: {reach['t_beg']:8.2f} -> {reach['t_end']:8.2f}, Duration: {reach['t_end'] - reach['t_beg']:5.2f}s, Wrist: {reach['wrist']:6s}, Travel: wrist {reach['wrist_travel']:.2f} m, shoulder: {reach['shoulder_travel']:.2f} m, ratio {reach['S/W ratio']:.2f}, "
    txt2 = f"  : {reach['t_beg_distance']:8.2f} -> {reach['t_end_distance']:8.2f}, Duration: {reach['t_end_distance'] - reach['t_beg_distance']:5.2f}s, Wrist: {reach['wrist']:6s}, Travel: wrist {reach['wrist_travel_new']:.2f} m, shoulder: {reach['shoulder_travel_new']:.2f} m, ratio {reach['S/W ratio new']:.2f}, "
    print(f"{txt1} \n{txt2} \n")

In [ ]:
print_items(reach)

In [ ]:
# plot the action zones with the SAU and MAU
fig_panu, ax = plt.subplots(figsize=(30, 15))

# plot the left and right wrist distances
ax.plot(
    kinect_mocap["time_stamps"],
    left_wrist_distance,
    ".",
    label=f"Left wrist ({'Paretic' if 'left' in paretic else 'Non-paretic'})",
    color="b",
)
ax.plot(
    kinect_mocap["time_stamps"],
    right_wrist_distance,
    ".",
    label=f"Right wrist ({'Paretic' if 'right' in paretic else 'Non-paretic'})",
    color="k",
)

# plot the left and right filtered wrist distances
for reach in reaches:
    if reach["wrist"] == "left":
        filtered_left_wrist_distance = reach["filtered_distance"]
        break
for reach in reaches:
    if reach["wrist"] == "right":
        filtered_right_wrist_distance = reach["filtered_distance"]
        break

ax.plot(
    kinect_mocap["time_stamps"],
    filtered_left_wrist_distance,
    ".-",
    markersize=0.5,
    linewidth=0.5,
    ##label="Left wrist distance (filtered)",
    color="b",
)

ax.plot(
    kinect_mocap["time_stamps"],
    filtered_right_wrist_distance,
    ".-",
    markersize=0.5,
    linewidth=0.5,
    # label="Right wrist distance (filtered)",
    color="k",
)


# plot the left and right shoulder distances
ax.plot(
    kinect_mocap["time_stamps"],
    left_shoulder_distance,
    ".",
    label="Left shoulder",
    color="b",
)
ax.plot(
    kinect_mocap["time_stamps"],
    right_shoulder_distance,
    ".",
    label="Right shoulder",
    color="k",
)

# plot the left and right filtered shoulder distances
filtered_left_shoulder_distance = lowpass_filter(
    kinect_mocap["time_stamps"],
    left_shoulder_distance,
    cutoff=2.5,
)
filtered_right_shoulder_distance = lowpass_filter(
    kinect_mocap["time_stamps"],
    right_shoulder_distance,
    cutoff=2.5,
)
ax.plot(
    kinect_mocap["time_stamps"],
    filtered_left_shoulder_distance,
    ".-",
    markersize=0.5,
    linewidth=0.5,
    # label="Left shoulder distance (filtered)",
    color="b",
)
ax.plot(
    kinect_mocap["time_stamps"],
    filtered_right_shoulder_distance,
    ".-",
    markersize=0.5,
    linewidth=0.5,
    # label="Right shoulder distance (filtered)",
    color="k",
)

# plot the action zones (start and stop times)
for i, (start, stop) in enumerate(zip(action_start_times, action_stop_times)):
    ax.vlines(
        start,
        ymin=np.min(left_wrist_distance),
        ymax=np.max(left_wrist_distance),
        color="g",
        linestyle="--",
    )
    ax.vlines(
        stop,
        ymin=np.min(left_wrist_distance),
        ymax=np.max(left_wrist_distance),
        color="r",
        linestyle="--",
    )
    # add the number of reaches in the zone
    ax.text(
        (start + stop) / 2,
        0.02,
        f"{action_nb_reaches_np[i]:02d}",
        ha="center",
        va="bottom",
        color=f"{'k' if 'left' in paretic else 'b'}",
        fontsize=10,
    )
    ax.text(
        (start + stop) / 2,
        0.0,
        f"{action_nb_reaches_p[i]:02d}",
        ha="center",
        va="bottom",
        color=f"{'b' if 'left' in paretic else 'k'}",
        fontsize=10,
    )
# add the SAU and MAU as a colored zone
top = np.max([reach["wrist_start_distance"] for reach in reaches]) + 0.01
bot = np.min([reach["wrist_end_distance"] for reach in reaches]) - 0.01


def plot_zone(ax, zone, color):
    """Plot a zone on the graph"""
    ax.fill_betweenx(
        [bot, top],
        zone["start time"],
        zone["end time"],
        color=color,
        alpha=0.2,
        label=zone["name"],
    )


plot_zone(ax, cal, "red")
plot_zone(ax, sau, "orange")
plot_zone(ax, mau, "black")

for i, reach in enumerate(reaches):

    if reach["wrist"] == "left":
        color = "b"
        wrist_data = left_wrist_distance
    else:
        color = "k"
        wrist_data = right_wrist_distance

    # a dotted line from the start to the end of the reach
    ax.plot(
        [reach["t_beg"], reach["t_end"]],
        [reach["wrist_start_distance"], reach["wrist_end_distance"]],
        color=color,
        linestyle="--",
        linewidth=0.5,
        alpha=0.2,
    )
    # a large star at the end
    ax.plot(
        reach["t_end"],
        reach["wrist_end_distance"],
        # make a large star with red border
        markerfacecolor=color,
        markeredgecolor="red",
        markersize=20,
        marker="*",
        # move to the front
        zorder=10,
        alpha=0.2,
    )
    # a large star at the start
    ax.plot(
        reach["t_beg"],
        reach["wrist_start_distance"],
        markerfacecolor=color,
        markeredgecolor="green",
        markersize=20,
        marker="*",
        zorder=10,
        alpha=0.2,
    )
    # plot the data used to compute the median
    # if reach["wrist"] == "left":
    #     wrist_data = left_wrist_distance
    # else:
    #     wrist_data = right_wrist_distance
    # beg_mask = range(reach["t_beg_i"] - 10, reach["t_beg_i"] + 10)
    # end_mask = range(reach["t_end_i"] - 10, reach["t_end_i"] + 10)
    # ax.plot(
    #     kinect_mocap["time_stamps"][beg_mask],
    #     wrist_data[beg_mask],
    #     "o",
    #     color="green",
    # )
    # ax.plot(
    #     kinect_mocap["time_stamps"][end_mask],
    #     wrist_data[end_mask],
    #     "o",
    #     color="red",
    # )

    # add the new limits for the wrists
    ax.plot(
        [reach["t_beg_distance"], reach["t_end_distance"]],
        [reach["wrist_start_new"], reach["wrist_end_new"]],
        color=color,
        linestyle="-.",
    )
    ax.plot(
        reach["t_beg_distance"],
        reach["wrist_start_new"],
        markerfacecolor="white",
        markeredgecolor="green",
        markersize=20,
        marker="*",
        zorder=10,
    )
    ax.plot(
        reach["t_end_distance"],
        reach["wrist_end_new"],
        markerfacecolor="white",
        markeredgecolor="red",
        markersize=20,
        marker="*",
        zorder=10,
    )

    # add the new limits for the shoulders

    ax.plot(
        reach["t_beg_distance"],
        reach["shoulder_start_new"],
        markerfacecolor="white",
        markeredgecolor="green",
        markersize=20,
        marker="*",
        zorder=10,
    )
    ax.plot(
        reach["t_end_distance"],
        reach["shoulder_end_new"],
        markerfacecolor="white",
        markeredgecolor="red",
        markersize=20,
        marker="*",
        zorder=10,
    )

    # print(f"Reach {i:02d}: {reach['t_beg']:.2f} -> {reach['t_end']:.2f} ( {kinect_mocap["time_stamps"][reach["t_beg_i"]]:.2f} -> {kinect_mocap["time_stamps"][reach["t_end_i"]]:.2f} )")

ylim = ax.get_ylim()
ax.set_ylim(0, ylim[1])
ax.legend()
ax.set_xlabel("Time (s)")
ax.set_ylabel("Distance to target (m)")
plt.show()

In [ ]:
# get the reaches in the SAU and MAU
def get_reaches_in_zone(reaches, zone):
    """Get the reaches in the zone"""
    reaches_in_zone = []
    for reach in reaches:
        if reach["t_beg"] >= zone["start time"] and reach["t_end"] <= zone["end time"]:
            reaches_in_zone.append(reach)
    return reaches_in_zone


sau_reaches = get_reaches_in_zone(reaches_panu, sau)
mau_reaches = get_reaches_in_zone(reaches_panu, mau)
# print the reaches in the SAU and MAU
print("SAU reaches:")
for i, reach in enumerate(sau_reaches):
    print(
        f"{i:02d}: {reach['t_beg']:8.2f} -> {reach['t_end']:8.2f}, Duration: {reach['t_end'] - reach['t_beg']:5.2f}s, ratio: {reach['S/W ratio']:.2f} {reach['wrist']}"
    )
print("MAU reaches:")
for i, reach in enumerate(mau_reaches):
    print(
        f"{i:02d}: {reach['t_beg']:8.2f} -> {reach['t_end']:8.2f}, Duration: {reach['t_end'] - reach['t_beg']:5.2f}s, Wrist: {reach['wrist']}"
    )

In [ ]:
# save the reaches in a CSV file
def save_reaches_in_csv(reaches, file_name):
    """Save the reaches in a CSV file"""
    # create the directory if it does not exist
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    # create the dataframe
    df = pd.DataFrame(reaches)
    # save the dataframe in a CSV file
    df.to_csv(file_name, index=False)


def save_reaches_in_csv_file_name(reaches, xdf_fullFname):
    """Save the reaches in a CSV file with the file name"""
    # get the file name
    file_name = xdf_fullFname.replace(".xdf", "_reaches.csv")
    save_reaches_in_csv(reaches, file_name)


def load_reaches_from_csv_file_name(xdf_fullFname):
    """Load the reaches from a CSV file with the file name"""
    # get the file name
    file_name = xdf_fullFname.replace(".xdf", "_reaches.csv")
    # load the dataframe from the CSV file
    df = pd.read_csv(file_name)
    # reaches = df.to_dict(orient="records")
    return df


# save the reaches in a CSV file
save_reaches_in_csv_file_name(reaches_left, xdf_fullFname)
save_reaches_in_csv_file_name(reaches_right, xdf_fullFname)

In [ ]:
print_items(reaches[0])

In [ ]:
# read the CSV file
df = load_reaches_from_csv_file_name(xdf_fullFname)

# get the median of the ration by condition
sau_reaches = df[df["condition"] == "sau"]
mau_reaches = df[df["condition"] == "mau"]
sau_median = sau_reaches["S/W ratio"].median()
mau_median = mau_reaches["S/W ratio"].median()

panu = sau_median - mau_median

sau_median_new = sau_reaches["S/W ratio new"].median()
mau_median_new = mau_reaches["S/W ratio new"].median()

panu_new = sau_median_new - mau_median_new

print(xdf_fullFname)
print(f"SAU median: {sau_median:.2f}")
print(f"MAU median: {mau_median:.2f}")
print(f"PANU: {panu:.2f}")

print(f"SAU median new: {sau_median_new:.2f}")
print(f"MAU median new: {mau_median_new:.2f}")
print(f"PANU new: {panu_new:.2f}")


plt.figure(fig_panu)
plt.title(
    f"{xdf_fullFname}: PANU: {panu:.2f}, SAU: {sau_median:.2f}, MAU: {mau_median:.2f} -- {panu_new:.2f}, SAU: {sau_median_new:.2f}, MAU: {mau_median_new:.2f}"
)
plt.tight_layout()
plt.show()

# # remove all figures but fig_panu
# figs = plt.get_fignums()
# for fig in figs:
#     if fig != fig_panu.number:
#         plt.close(fig)

# save the figure
fig_fullFname = xdf_fullFname.replace(".xdf", "_xdf_PANU.png")
fig_panu.savefig(fig_fullFname, dpi=300)